This notebook takes the product of `dap_job_quality/pipeline/find_job_quality.py` and carries out some simple EDA. Some of these graphs were used in the ONS checkpoint meeting on 25.06.2024 (see [these slides](https://docs.google.com/presentation/d/1zPXDAyvUg55waYRdwDy_nei37lDcf0Yi1dA-wdZXp6k/edit#slide=id.g2e7cc68b5f6_0_0))

In [ ]:
import altair as alt
# import altair_saver as saver
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt

# Download required nltk resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

from dap_job_quality import BUCKET_NAME, PROJECT_DIR
from dap_job_quality.getters.data_getters import load_s3_data
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.getters.afs_data import get_stratified_sample

pd.set_option('display.width', 1000)  # Set the width to 1000 characters
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', 100)  # Set the max column width if needed

In [ ]:
def clean_and_lemmatize(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [lemmatizer.lemmatize(token) for token in tokens if token.isalpha() and token not in stop_words]
    return ' '.join(filtered_tokens)

In [ ]:
LOOKUP = get_keywords()[['target_phrase','subcategory','dimension']]

In [ ]:
afs_data = load_s3_data(BUCKET_NAME, "job_quality/early_years/evaluation_sample/job_ads_prod_True_sample_10000_2024-06-20.parquet")

In [ ]:
afs_data = afs_data.merge(LOOKUP, left_on='target_phrase', right_on='target_phrase', how='left')
afs_data['cleaned_sentences'] = afs_data['sentences_split'].apply(clean_and_lemmatize)
afs_data.head()

In [ ]:
# Function to create word cloud for each subcategory
def create_wordcloud(subcategory, df=afs_data):
    text = ' '.join(df[df['subcategory'] == subcategory]['cleaned_sentences'])
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Subcategory: {subcategory}')
    plt.show()

# Create word clouds for each distinct value of 'subcategory'
for subcategory in afs_data['subcategory'].unique():
    create_wordcloud(subcategory)

In [ ]:
afs_data['subcategory'].value_counts()

In [ ]:
target_dimensions = ['FLEX_LOC', 'FLEX_HOURS', 'HOURS', 'L&D']

In [ ]:
afs_data

In [ ]:
def make_barplot(df=afs_data, dim='FLEX_LOC'):
    subset = df[df['subcategory']==dim]
    
    # Group by 'target_phrase' and get the counts
    counts_df = subset['target_phrase'].value_counts().reset_index()
    counts_df.columns = ['target_phrase', 'counts']

    plt.figure(figsize=(10, 8))
    sns.barplot(x='counts', y='target_phrase', data=counts_df, palette='viridis')
    plt.xlabel('Counts', fontsize=14)
    plt.ylabel('Target Phrase', fontsize=14)
    plt.title(f'Target Phrase Counts for {dim}', fontsize=16)

    for index, value in enumerate(counts_df['counts']):
        plt.text(value, index, str(value), color='black', ha="left", va='center', fontsize=12)

    plt.savefig(PROJECT_DIR / 'outputs' / 'figures' / f'target_phrase_counts_{dim}.png', bbox_inches='tight')
    plt.close()

In [ ]:
for dim in target_dimensions:
    make_barplot(df=afs_data, dim=dim)

In [ ]:
stratified_sample = get_stratified_sample()
stratified_sample.columns

In [ ]:
stratified_sample = stratified_sample.merge(afs_data, on=['id', 'clean_description'], how='left')
stratified_sample.head()

In [ ]:
stratified_sample[stratified_sample['subcategory'] == 'FLEX_HOURS'][['occupation', 'year', 'sentences_split', 'ngrams','target_phrase', 'cosine_similarity', 'subcategory']].sort_values('cosine_similarity', ascending=False).to_csv('example_matches.csv')

In [ ]:
stratified_sample.columns

In [ ]:
len(stratified_sample['id'].unique())

In [ ]:
# Calculate the number of unique IDs
total_ids = stratified_sample['id'].nunique()

# Calculate the number of unique IDs that contain each target phrase
phrase_counts = stratified_sample.groupby(['subcategory','target_phrase'])['id'].nunique().reset_index(name='unique_id_count')

# Calculate the proportion of IDs for each target phrase
phrase_counts['proportion'] = phrase_counts['unique_id_count'] / total_ids

In [ ]:
phrase_counts['percentage'] = phrase_counts['proportion'] * 100
phrase_counts

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(x='percentage', y='target_phrase', data=phrase_counts[phrase_counts['subcategory']=='FLEX_HOURS'], palette='viridis')

for index, value in enumerate(phrase_counts[phrase_counts['subcategory']=='FLEX_HOURS']['percentage']):
    plt.text(value, index, str(round(value, 2)), color='black', ha="left", va='center', fontsize=12)

plt.savefig(PROJECT_DIR / 'outputs' / 'figures' / f'target_phrase_perc_FLEX_HOURS.png', bbox_inches='tight')
plt.close()

In [ ]:
stratified_sample[stratified_sample['target_phrase']=='compressed hours']

In [ ]:
bar_chart = alt.Chart(phrase_counts[phrase_counts['subcategory']=='L&D']).mark_bar().encode(
    x=alt.X('target_phrase:N', title='Target Phrase'),
    y=alt.Y('proportion:Q', title='Proportion of IDs'),
    tooltip=['target_phrase', 'proportion']
).properties(
    width=600,
    height=400,
    title='Proportion of IDs Containing Each Target Phrase'
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16
)

bar_chart.display()